# 07: Validate metadata-recovery pilot

Independent checks of the four attached files only:

- `audit/metadata_pilot_sample_sizes.json`
- `audit/metadata_pilot_sample.csv`
- `audit/metadata_pilot_link_status.csv`
- `audit/metadata_pilot_observations.csv`

Does **not** modify V1. Does **not** choose V2 eligibility, conflict aggregation, or similarity thresholds. Does **not** retrieve the full pool.

These observations are IEDB at retrieval date 2026-11-07, not the 2025-11-03 export. Missingness is kept as a finding.

In [1]:
from collections import Counter, defaultdict
from pathlib import Path
import json
import pandas as pd

AUDIT = Path("..") / "audit"
sizes = json.loads((AUDIT / "metadata_pilot_sample_sizes.json").read_text())
sample = pd.read_csv(AUDIT / "metadata_pilot_sample.csv", dtype=str).fillna("")
links = pd.read_csv(AUDIT / "metadata_pilot_link_status.csv", dtype=str).fillna("")
obs = pd.read_csv(AUDIT / "metadata_pilot_observations.csv", dtype=str).fillna("")
for col in ["n_assays", "n_epitope_ids_queried", "linked"]:
    if col in links.columns:
        links[col] = pd.to_numeric(links[col], errors="coerce")
print("sample", sample.shape, "links", links.shape, "obs", obs.shape)
print("sizes keys", list(sizes))

sample (46, 15) links (46, 10) obs (1110, 24)
sizes keys ['retained_pos_iedb_only', 'retained_neg_iedb_only', 'retained_both_databases', 'conflict_iedb_only', 'conflict_both_databases', 'cedar_only']


## 1. Structure, IDs, and linkage

In [2]:
def nonempty(s):
    return s.fillna("").astype(str).str.strip().ne("")

print("unique peptides sample", sample["peptide"].nunique(), "dups", sample["peptide"].duplicated().sum())
print("sample peptides == link peptides (order)", list(sample["peptide"]) == list(links["peptide"]))
print("obs peptides not in sample", sorted(set(obs["export_peptide"]) - set(sample["peptide"])))
print("sample peptides not in obs", sorted(set(sample["peptide"]) - set(obs["export_peptide"])))
print("unique observation_id", obs["observation_id"].nunique(), "unique tcell_id", obs["tcell_id"].nunique())
print("sum(links.n_assays)", int(links["n_assays"].sum()), "n_obs", len(obs))
cmp = links.merge(obs.groupby("export_peptide").size().rename("obs_n"), left_on="peptide", right_index=True, how="left")
print("link vs obs count mismatches", int((cmp["n_assays"] != cmp["obs_n"]).sum()))
print("linked==1", int((links["linked"] == 1).sum()), "nonempty errors", int(nonempty(links["errors"]).sum()))

unique peptides sample 46 dups 0
sample peptides == link peptides (order) True
obs peptides not in sample []
sample peptides not in obs []
unique observation_id 1110 unique tcell_id 1110
sum(links.n_assays) 1110 n_obs 1110
link vs obs count mismatches 0
linked==1 46 nonempty errors 0


In [3]:
sample_counts = sample["pilot_stratum"].value_counts().to_dict()
print("sample stratum counts", sample_counts)
print("json n_sampled", {k: v["n_sampled"] for k, v in sizes.items()})
print("sequence_matches_export != 1", int((obs["sequence_matches_export"] != "1").sum()))
print("linear_sequence != export_peptide", int((obs["linear_sequence"] != obs["export_peptide"]).sum()))
print("retrieval_date", sorted(obs["retrieval_date"].unique()))
print("retrieval_source", sorted(obs["retrieval_source"].unique()))
print("multi epitope_ids", int((sample["epitope_ids"].str.contains("\\|")).sum()))

sample stratum counts {'retained_pos_iedb_only': 10, 'retained_neg_iedb_only': 10, 'retained_both_databases': 10, 'conflict_iedb_only': 8, 'conflict_both_databases': 8}
json n_sampled {'retained_pos_iedb_only': 10, 'retained_neg_iedb_only': 10, 'retained_both_databases': 10, 'conflict_iedb_only': 8, 'conflict_both_databases': 8, 'cedar_only': 0}
sequence_matches_export != 1 0
linear_sequence != export_peptide 0
retrieval_date ['2026-11-07T05:18:06Z']
retrieval_source ['iedb_query_api.tcell_search']
multi epitope_ids 0


## 2. Coverage: nonempty vs specific

In [4]:
fields = [
    "qualitative_measure", "assay_names", "host_organism_name", "mhc_class",
    "mhc_allele_name", "mhc_restriction", "mhc_allele_evidence", "reference_id",
    "pubmed_id", "immunization_description", "parent_source_antigen_name", "source_organism_name",
]
rows = []
for f in fields:
    n = int(nonempty(obs[f]).sum())
    pep = int(obs.loc[nonempty(obs[f]), "export_peptide"].nunique())
    rows.append({"field": f, "assay_nonempty": n, "assay_frac": round(n / len(obs), 4), "peptides_with_any": pep})
pd.DataFrame(rows)

,field,assay_nonempty,assay_frac,peptides_with_any
0,qualitative_measure,1110,1.0000,46
1,assay_names,1110,1.0000,46
2,host_organism_name,1110,1.0000,46
3,mhc_class,1104,0.9946,44
4,mhc_allele_name,1104,0.9946,44
5,mhc_restriction,1104,0.9946,44
6,mhc_allele_evidence,1104,0.9946,44
7,reference_id,1110,1.0000,46
8,pubmed_id,1084,0.9766,40
9,immunization_description,1097,0.9883,46


In [5]:
print("qualitative_measure\n", obs["qualitative_measure"].value_counts())
print("\nmhc_class\n", obs["mhc_class"].replace("", "MISSING").value_counts())
print("\nmhc_allele_evidence\n", obs["mhc_allele_evidence"].replace("", "MISSING").value_counts())
coarse_allele = obs["mhc_allele_name"].isin(["human", "HLA class I", "HLA class II", ""])
print("\ncoarse or missing allele_name", int(coarse_allele.sum()))
print("host value counts")
print(obs["host_organism_name"].value_counts().head(15))
print("\ntop assay_names")
print(obs["assay_names"].value_counts().head(15))

qualitative_measure
 qualitative_measure
Positive                 959
Negative                  93
Positive-Low              39
Positive-High             17
Positive-Intermediate      2
Name: count, dtype: int64

mhc_class
 mhc_class
I          1092
II           12
MISSING       6
Name: count, dtype: int64

mhc_allele_evidence
 mhc_allele_evidence
Cited reference                               804
Single allele present                         124
MHC binding assay                              74
Not determined                                 42
MHC binding prediction                         22
T cell assay -Mismatched MHC molecules         18
T cell assay -T cell subset identification     13
MISSING                                         6
Inferred by motif or alleles present            4
T cell assay -Biological process measured       2
T cell assay -MHC subset identification         1
Name: count, dtype: int64

coarse or missing allele_name 21
host value counts
host_organism_name
Hom

## 3. Concentration and disagreement

Context key for disagreement: host + allele + assay_names + reference_id. Empty parts mean the context is incomplete (unknown), not that settings are identical.

In [6]:
n_by_pep = obs.groupby("export_peptide").size().sort_values(ascending=False)
print(n_by_pep.head(8))
top = n_by_pep.index[0]
print("top peptide", top, int(n_by_pep.iloc[0]), "frac", round(n_by_pep.iloc[0] / len(obs), 4))
print("assays by stratum\n", obs.groupby("pilot_stratum").size())
print("excluding top\n", obs[obs["export_peptide"] != top].groupby("pilot_stratum").size())
print("n_obs excluding top", int((obs["export_peptide"] != top).sum()))
print("n_reference excluding top", obs.loc[obs["export_peptide"] != top, "reference_id"].nunique())
print("n_reference all", obs["reference_id"].nunique())

export_peptide
NLVPMVATV     920
LYCESVHNF      48
ALWGFFPVL      27
HPMPINIKL      21
DYKHYTPSF       6
DLIDEYITWR      6
RVLTGWGLN       5
YMWLGARY        5
dtype: int64
top peptide NLVPMVATV 920 frac 0.8288
assays by stratum
 pilot_stratum
conflict_both_databases    988
conflict_iedb_only          56
retained_both_databases     11
retained_neg_iedb_only      22
retained_pos_iedb_only      33
dtype: int64
excluding top
 pilot_stratum
conflict_both_databases    68
conflict_iedb_only         56
retained_both_databases    11
retained_neg_iedb_only     22
retained_pos_iedb_only     33
dtype: int64
n_obs excluding top 190
n_reference excluding top 74
n_reference all 430


In [7]:
def is_pos(s):
    return s.str.lower().str.startswith("positive")

def is_neg(s):
    return s.eq("Negative")

rows = []
for pep, g in obs.groupby("export_peptide"):
    rows.append({
        "peptide": pep,
        "stratum": g["pilot_stratum"].iloc[0],
        "v1_conflict": g["v1_conflict"].iloc[0],
        "v1_query_label": g["v1_query_label"].iloc[0],
        "n_pos": int(is_pos(g["qualitative_measure"]).sum()),
        "n_neg": int(is_neg(g["qualitative_measure"]).sum()),
        "n": len(g),
    })
pep = pd.DataFrame(rows)
print("retained pos with any current neg", int(((pep.v1_conflict=="0") & (pep.v1_query_label=="1") & (pep.n_neg>0)).sum()))
print("retained neg with any current pos", int(((pep.v1_conflict=="0") & (pep.v1_query_label=="0") & (pep.n_pos>0)).sum()))
print("conflict with both pos and neg now", int(((pep.v1_conflict=="1") & (pep.n_pos>0) & (pep.n_neg>0)).sum()), "of", int((pep.v1_conflict=="1").sum()))

retained pos with any current neg 0
retained neg with any current pos 0
conflict with both pos and neg now 16 of 16


In [8]:
within = across = incomplete = 0
for pep, g in obs.groupby("export_peptide"):
    pos = g[is_pos(g["qualitative_measure"])]
    neg = g[is_neg(g["qualitative_measure"])]
    if pos.empty or neg.empty:
        continue
    def ctx(df):
        return set(zip(df["host_organism_name"], df["mhc_allele_name"], df["assay_names"], df["reference_id"]))
    overlap = ctx(pos) & ctx(neg)
    if not overlap:
        across += 1
    elif any(not all(part.strip() for part in c) for c in overlap):
        incomplete += 1
    else:
        within += 1
print("peptides with both pos and neg assays", within + across + incomplete)
print("within observed context", within)
print("across context", across)
print("overlap but incomplete context", incomplete)
print("peptides with n_assays>=1000", int((n_by_pep >= 1000).sum()), ">=500", int((n_by_pep >= 500).sum()))

peptides with both pos and neg assays 16
within observed context 5
across context 11
overlap but incomplete context 0
peptides with n_assays>=1000 0 >=500 1


## 4. What this pilot can and cannot support

Supported: original epitope IDs can be linked to current IEDB T-cell assay rows; required fields are often present; V1 conflicts still show mixed outcomes in current IEDB; one peptide can dominate assay and study counts.

Not supported: exhaustive API completeness (no page headers saved); population prevalence; V2 filters; study-held-out feasibility from raw study counts alone; similarity-audit claims (those files were not part of this four-file check).